In [2]:
# Load libraries
library("anndata")
library("DESeq2")
library("reticulate")

In [3]:
# Use the path to your Python executable in the virtual environment
use_python("/lila/home/forsythb/.virtualenvs/r-reticulate/bin/")

# Look at where Python is located
py_config()

# Import scanpy
sc <- import("scanpy")

python:         /lila/home/forsythb/.virtualenvs/r-reticulate/bin/python
libpython:      /home/forsythb/anaconda3/lib/libpython3.8.so
pythonhome:     /lila/home/forsythb/.virtualenvs/r-reticulate:/lila/home/forsythb/.virtualenvs/r-reticulate
version:        3.8.18 (default, Sep 11 2023, 13:40:15)  [GCC 11.2.0]
numpy:          /lila/home/forsythb/.virtualenvs/r-reticulate/lib/python3.8/site-packages/numpy
numpy_version:  1.24.4

NOTE: Python version was forced by use_python() function

In [4]:
# Read h5ad file
adata <- sc$read_h5ad("/data/chanjlab/CRC_ZFP36L2.092023/Organoid/output/combined/out_post_hvg/adata.filteredmultiplex.combined.hvg_5000.h5ad")

In [5]:
adata

AnnData object with n_obs × n_vars = 59140 × 32485
    obs: 'background_fraction', 'cell_probability', 'cell_size', 'droplet_efficiency', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'pct_counts_in_top_50_genes', 'pct_counts_in_top_100_genes', 'pct_counts_in_top_200_genes', 'pct_counts_in_top_500_genes', 'log10GenesPerUMI', 'original_total_counts', 'log10_original_total_counts', 'mito_frac', 'Patient', 'Tumor_Site', 'Culture_Media', 'ZFP_Expression', 'Replicate', 'Batch', 'Sample', 'phenograph'
    var: 'n_cells_by_counts', 'mean_counts', 'log1p_mean_counts', 'pct_dropout_by_counts', 'total_counts', 'log1p_total_counts', 'ribo', 'n_cells', 'highly_variable', 'highly_variable_rank', 'means', 'variances', 'variances_norm', 'highly_variable_nbatches'
    uns: 'hvg', 'log1p', 'neighbors', 'num_components', 'paga', 'phenograph_sizes', 'umap', 'var_explained'
    obsm: 'X_pca', 'X_umap', 'gene_expression_encoding'
    layers: 'counts', 'without_log'
 

In [6]:
# Extract count matrix from the raw layer
count_matrix <- adata$raw$X
count_matrix <- t(count_matrix)
count_matrix <- round(count_matrix)

In [18]:
dim(count_matrix)

[1] 32485 59140

In [26]:
sum(rowSums(count_matrix>10)<=10)

[1] 32479

In [24]:
rowSums(count_matrix>10)

A1BG          A1BG-AS1              A1CF               A2M 
                0                 0                 0                 0 
          A2M-AS1             A2ML1         A2ML1-AS1         A2ML1-AS2 
                0                 0                 0                 0 
          A3GALT2            A4GALT             A4GNT              AAAS 
                0                 0                 0                 0 
             AACS             AADAC           AADACL2       AADACL2-AS1 
                0                 0                 0                 0 
          AADACL3           AADACL4             AADAT             AAGAB 
                0                 0                 0                 0 
             AAK1             AAMDC              AAMP             AANAT 
                0                 0                 0                 0 
             AAR2              AARD              AARS             AARS2 
                0                 0                 0                 0 
           AARSD1             AASDH          AASDHPPT              AASS 
                0                 0                 0                 0 
            AATBC              AATF              AATK            ABALON 
                0                 0                 0                 0 
             ABAT             ABCA1            ABCA10            ABCA12 
                0                 0                 0                 0 
           ABCA13             ABCA2             ABCA3             ABCA4 
                0                 0                 0                 0 
            ABCA5             ABCA6             ABCA7             ABCA8 
                0                 0                 0                 0 
            ABCA9         ABCA9-AS1             ABCB1            ABCB10 
                0                 0                 0                 0 
           ABCB11             ABCB4             ABCB5             ABCB6 
                0                 0                 0                 0 
            ABCB7             ABCB8             ABCB9             ABCC1 
                0                 0                 0                 0 
           ABCC10            ABCC11            ABCC12             ABCC2 
                0                 0                 0                 0 
            ABCC3             ABCC4             ABCC5         ABCC5-AS1 
                0                 0                 0                 0 
            ABCC6             ABCC8             ABCC9             ABCD1 
                0                 0                 0                 0 
            ABCD2             ABCD3             ABCD4             ABCE1 
                0                 0                 0                 0 
            ABCF1             ABCF2             ABCF3             ABCG1 
                0                 0                 0                 0 
            ABCG2             ABCG4             ABCG5             ABCG8 
                0                 0                 0                 0 
            ABHD1            ABHD10            ABHD11            ABHD12 
                0                 0                 0                 0 
          ABHD12B            ABHD13           ABHD14A           ABHD14B 
                0                 0                 0                 0 
           ABHD15        ABHD15-AS1           ABHD16A           ABHD16B 
                0                 0                 0                 0 
          ABHD17A           ABHD17B           ABHD17C            ABHD18 
                0                 0                 0                 0 
            ABHD2             ABHD3             ABHD4             ABHD5 
                0                 0                 0                 0 
            ABHD6             ABHD8              ABI1              ABI2 
                0                 0                 0                 0 
             ABI3            ABI3BP           ABITRAM              ABL1

In [8]:
# Define the column data
coldata <- adata$obs[,c("Tumor_Site","Culture_Media", "ZFP_Expression", "Batch", "Patient")]

# Convert column data to factors
coldata$Tumor_Site <- factor(coldata$Tumor_Site)
coldata$Culture_Media <- factor(coldata$Culture_Media)
coldata$ZFP_Expression <- factor(coldata$ZFP_Expression)
coldata$Batch <- factor(coldata$Batch)
coldata$Patient <- factor(coldata$Patient)

In [9]:
coldata

,Tumor_Site,Culture_Media,ZFP_Expression,Batch,Patient
,<fct>,<fct>,<fct>,<fct>,<fct>
146P_BASE_shZFP36L2_3_TTAGGGTCAGTAACGG-1,Primary,BASE,ZFP_KD,5,146
KG146Li_BASE_shZFP36L2_4_CGTTAGACACGTAACT-1,Metastatic,BASE,ZFP_KD,2,146
146P_BASE_shCTRL_AGGTGTTCATACAGGG-1,Primary,BASE,CTRL,5,146
146P_BASE_shZFP36L2_4_CCTCTAGGTCGCATTA-1,Primary,BASE,ZFP_KD,5,146
125P_HISC_shZFP36L2_3_GTTGTCCTCCACTGGG-1,Primary,HISC,ZFP_KD,1,125
146P_BASE_shZFP36L2_4_CTGTAGACACTCAAGT-1,Primary,BASE,ZFP_KD,5,146
KG146Li_BASE_shCtrl_TTGTTCACAGGAGACT-1,Metastatic,BASE,CTRL,2,146
KG146Li_BASE_shZFP36L2_3_TCTTGCGAGCATGATA-1,Metastatic,BASE,ZFP_KD,2,146
KG146Li_BASE_shZFP36L2_4_TACTTACAGAGTCCGA-1,Metastatic,BASE,ZFP_KD,2,146


In [10]:
# Design a DESeqDataSet
dds_1 <- DESeqDataSetFromMatrix(
  countData = count_matrix,
  colData = coldata,
  design = ~ Tumor_Site + Culture_Media + Patient + ZFP_Expression
)

converting counts to integer mode



In [11]:
# Add a pseudo-count of 1 to all counts
count_matrix <- counts(dds_1) + 1

# Specify the row names as the gene names
rownames(count_matrix) <- adata$raw$var_names

In [12]:
count_matrix

,146P_BASE_shZFP36L2_3_TTAGGGTCAGTAACGG-1,KG146Li_BASE_shZFP36L2_4_CGTTAGACACGTAACT-1,146P_BASE_shCTRL_AGGTGTTCATACAGGG-1,146P_BASE_shZFP36L2_4_CCTCTAGGTCGCATTA-1,125P_HISC_shZFP36L2_3_GTTGTCCTCCACTGGG-1,146P_BASE_shZFP36L2_4_CTGTAGACACTCAAGT-1,KG146Li_BASE_shCtrl_TTGTTCACAGGAGACT-1,KG146Li_BASE_shZFP36L2_3_TCTTGCGAGCATGATA-1,KG146Li_BASE_shZFP36L2_4_TACTTACAGAGTCCGA-1,KG146Li_BASE_shZFP36L2_3_GCACATAGTAGTTCCA-1,⋯,146P_BASE_shZFP36L2_4_TTTGGTTTCGAAGCAG-1,125P_HISC_shZFP36L2_4_TACTTGTCAAGATGGC-1,146Li_dedifferentiation_shZFP36L2_4_GCAACATTCTTCTCAA-1,146P_HISC_shZFP36L2_4_ATGAGGGGTCACTGAT-1,125P_HISC_shCTRL_CGGACACCACAAAGTA-1,125P_HISC_shZFP36L2_4_TTCGGTCTCCGTTGGG-1,146P_HISC_shZFP36L2_4_ACAAAGATCATCACTT-1,146P_BASE_shZFP36L2_3_TCAGGTAAGTTTAGGA-1,146P_HISC_shZFP36L2_3_TCATCATAGATCGGTG-1,125P_HISC_shZFP36L2_4_TCATCATCAGGTACGA-1
A1BG,1,1,1,1,1,1,1,1,1,1,⋯,1,1,1,1,1,1,1,1,1,1
A1BG-AS1,1,1,1,1,1,1,1,1,1,1,⋯,1,1,1,1,1,1,1,1,1,1
A1CF,1,4,3,1,1,2,3,1,1,2,⋯,4,1,1,1,1,1,1,2,1,1
A2M,1,1,1,1,1,1,1,1,1,1,⋯,1,1,1,1,1,1,1,1,1,1
A2M-AS1,1,1,1,1,1,1,1,1,1,1,⋯,1,1,1,1,1,1,1,1,1,1
A2ML1,1,1,1,1,1,1,1,1,1,1,⋯,1,1,1,1,1,1,1,1,1,1
A2ML1-AS1,1,1,1,1,1,1,1,1,1,1,⋯,1,1,1,1,1,1,1,1,1,1
A2ML1-AS2,1,1,1,1,1,1,1,1,1,1,⋯,1,1,1,1,1,1,1,1,1,1
A3GALT2,1,1,1,1,1,1,1,1,1,1,⋯,1,1,1,1,1,1,1,1,1,1
A4GALT,1,1,1,1,1,1,1,1,1,1,⋯,1,1,1,1,1,1,1,1,1,1


In [14]:
# Set the reference levels
coldata$Tumor_Site<-relevel(coldata$Tumor_Site,ref="Primary")
coldata$Culture_Media<-relevel(coldata$Culture_Media,ref="BASE")
coldata$ZFP_Expression<-relevel(coldata$ZFP_Expression,ref="CTRL")
coldata$Patient<-relevel(coldata$Patient,ref="125")

In [15]:
coldata

,Tumor_Site,Culture_Media,ZFP_Expression,Batch,Patient
,<fct>,<fct>,<fct>,<fct>,<fct>
146P_BASE_shZFP36L2_3_TTAGGGTCAGTAACGG-1,Primary,BASE,ZFP_KD,5,146
KG146Li_BASE_shZFP36L2_4_CGTTAGACACGTAACT-1,Metastatic,BASE,ZFP_KD,2,146
146P_BASE_shCTRL_AGGTGTTCATACAGGG-1,Primary,BASE,CTRL,5,146
146P_BASE_shZFP36L2_4_CCTCTAGGTCGCATTA-1,Primary,BASE,ZFP_KD,5,146
125P_HISC_shZFP36L2_3_GTTGTCCTCCACTGGG-1,Primary,HISC,ZFP_KD,1,125
146P_BASE_shZFP36L2_4_CTGTAGACACTCAAGT-1,Primary,BASE,ZFP_KD,5,146
KG146Li_BASE_shCtrl_TTGTTCACAGGAGACT-1,Metastatic,BASE,CTRL,2,146
KG146Li_BASE_shZFP36L2_3_TCTTGCGAGCATGATA-1,Metastatic,BASE,ZFP_KD,2,146
KG146Li_BASE_shZFP36L2_4_TACTTACAGAGTCCGA-1,Metastatic,BASE,ZFP_KD,2,146


In [16]:
# Design a DESeqDataSet with modified count matrix
dds_1 <- DESeqDataSetFromMatrix(
  countData = count_matrix,
  colData = coldata,
  design = ~ Tumor_Site + Culture_Media + Patient + ZFP_Expression 
)

converting counts to integer mode



In [17]:
dds_1

class: DESeqDataSet 
dim: 32485 59140 
metadata(1): version
assays(1): counts
rownames(32485): A1BG A1BG-AS1 ... ZYX ZZEF1
rowData names(0):
colnames(59140): 146P_BASE_shZFP36L2_3_TTAGGGTCAGTAACGG-1
  KG146Li_BASE_shZFP36L2_4_CGTTAGACACGTAACT-1 ...
  146P_HISC_shZFP36L2_3_TCATCATAGATCGGTG-1
  125P_HISC_shZFP36L2_4_TCATCATCAGGTACGA-1
colData names(5): Tumor_Site Culture_Media ZFP_Expression Batch Patient

In [1]:
# Set the CRAN mirror
options(repos = c(CRAN = "https://cran.r-project.org"))

# If not already downloaded in environment, then download DESeq2
if (!requireNamespace("BiocManager", quietly = TRUE)) {
  install.packages("BiocManager")
}
BiocManager::install("DESeq2")

'getOption("repos")' replaces Bioconductor standard repositories, see
'help("repositories", package = "BiocManager")' for details.
Replacement repositories:
    CRAN: https://cran.r-project.org

Bioconductor version 3.18 (BiocManager 1.30.22), R 4.3.1 (2023-06-16)

Warning message:
“package(s) not installed when version(s) same as or greater than current; use
  `force = TRUE` to re-install: 'DESeq2'”
Old packages: 'cli', 'cluster', 'evaluate', 'fansi', 'foreign', 'htmltools',
  'jsonlite', 'lifecycle', 'Matrix', 'nlme', 'rlang', 'rpart', 'rprojroot',
  'vctrs', 'withr'



In [1]:
library("anndata")
library("DESeq2")

Loading required package: S4Vectors

Loading required package: stats4

Loading required package: BiocGenerics


Attaching package: ‘BiocGenerics’


The following objects are masked from ‘package:stats’:

    IQR, mad, sd, var, xtabs


The following objects are masked from ‘package:base’:

    anyDuplicated, aperm, append, as.data.frame, basename, cbind,
    colnames, dirname, do.call, duplicated, eval, evalq, Filter, Find,
    get, grep, grepl, intersect, is.unsorted, lapply, Map, mapply,
    match, mget, order, paste, pmax, pmax.int, pmin, pmin.int,
    Position, rank, rbind, Reduce, rownames, sapply, setdiff, sort,
    table, tapply, union, unique, unsplit, which.max, which.min



Attaching package: ‘S4Vectors’


The following object is masked from ‘package:utils’:

    findMatches


The following objects are masked from ‘package:base’:

    expand.grid, I, unname


Loading required package: IRanges

Loading required package: GenomicRanges

Loading required package: GenomeInfoDb

Loa

In [2]:
# Read h5ad file
adata <- read_h5ad("/data/chanjlab/CRC_ZFP36L2.092023/Organoid/output/combined/out_post_hvg/adata.combined.hvg_5000.h5ad")

In [3]:
# Extract count matrix from the raw layer
count_matrix <- adata$raw$X
count_matrix <- t(count_matrix)
count_matrix <- round(count_matrix)

In [4]:
# Define the column data
coldata <- adata$obs[,c("Tumor_Site","Culture_Media", "ZFP_Expression", "Batch", "Patient")]

# Convert column data to factors
coldata$Tumor_Site <- factor(coldata$Tumor_Site)
coldata$Culture_Media <- factor(coldata$Culture_Media)
coldata$ZFP_Expression <- factor(coldata$ZFP_Expression)
coldata$Batch <- factor(coldata$Batch)
coldata$Patient <- factor(coldata$Patient)

In [5]:
# Design a DESeqDataSet
dds_1 <- DESeqDataSetFromMatrix(
  countData = count_matrix,
  colData = coldata,
  design = ~ Tumor_Site + Culture_Media + Patient + ZFP_Expression
)

# Add a pseudo-count of 1 to all counts
count_matrix <- counts(dds_1) + 1

# Specify the row names as the gene names
rownames(count_matrix) <- adata$raw$var_names


converting counts to integer mode



In [6]:
# Set the reference levels
dds_1$Tumor_Site<-relevel(dds_1$Tumor_Site,ref="Primary")
dds_1$Culture_Media<-relevel(dds_1$Culture_Media,ref="BASE")
dds_1$ZFP_Expression<-relevel(dds_1$ZFP_Expression,ref="CTRL")
dds_1$Patient<-relevel(dds_1$Patient,ref="125")

In [7]:
# Design a DESeqDataSet with modified count matrix
dds_1 <- DESeqDataSetFromMatrix(
  countData = count_matrix,
  colData = coldata,
  design = ~ Tumor_Site + Culture_Media + Patient + ZFP_Expression 
)

converting counts to integer mode



In [ ]:
# Perform DESeq analysis
dds_1 <- DESeq(dds_1)

estimating size factors

estimating dispersions

gene-wise dispersion estimates



In [ ]:
# Extract results
result_names_1 <- resultsNames(dds_1)
result_names_1

# Specify the contrast and extract results
res_1 <- results(dds_1)
res_1

In [ ]:
# Specify the file path where you want to save the results
result_file <- "/data/chanjlab/CRC_ZFP36L2.092023/Organoid/output/deseq/zfpexp_results.csv"

# Write the results table to a CSV file
write.csv(as.data.frame(res_1), file = result_file)

# Save the DESeqDataSet object
saveRDS(dds_1, file = "/data/chanjlab/CRC_ZFP36L2.092023/Organoid/output/deseq/zfpexp_dds.rds")

# Save the DESeq results
saveRDS(results(dds_1), file = "/data/chanjlab/CRC_ZFP36L2.092023/Organoid/output/deseq/zfpexp_results.rds")

In [1]:
# Install the R anndata package
install.packages("anndata")

# Run this only if you do not already have an installation of miniconda
#reticulate::install_miniconda(force = TRUE)
#install.packages("reticulate")

# Install the Python anndata package
#anndata::install_anndata()

In [13]:
#install.packages("reticulate")

In [1]:
#Sys.setenv(PATH = paste("lila/home/forsythb/anaconda3/envs/r_4.3.1/bin", Sys.getenv("PATH"), sep = ":"))

In [3]:
#library(reticulate)
#use_python("lila/home/forsythb/anaconda3/envs/r_4.3.1/bin/python")

In [2]:
# Load libraries
library(DESeq2)
library(anndata)

Loading required package: S4Vectors

Loading required package: stats4

Loading required package: BiocGenerics


Attaching package: ‘BiocGenerics’


The following objects are masked from ‘package:stats’:

    IQR, mad, sd, var, xtabs


The following objects are masked from ‘package:base’:

    anyDuplicated, aperm, append, as.data.frame, basename, cbind,
    colnames, dirname, do.call, duplicated, eval, evalq, Filter, Find,
    get, grep, grepl, intersect, is.unsorted, lapply, Map, mapply,
    match, mget, order, paste, pmax, pmax.int, pmin, pmin.int,
    Position, rank, rbind, Reduce, rownames, sapply, setdiff, sort,
    table, tapply, union, unique, unsplit, which.max, which.min



Attaching package: ‘S4Vectors’


The following object is masked from ‘package:utils’:

    findMatches


The following objects are masked from ‘package:base’:

    expand.grid, I, unname


Loading required package: IRanges

Loading required package: GenomicRanges

Loading required package: GenomeInfoDb

Loa

In [3]:
# Read h5ad file
adata <- read_h5ad("/data/chanjlab/CRC_ZFP36L2.092023/Organoid/output/combined/out_post_hvg/adata.combined.hvg_5000.h5ad")

In [4]:
# Extract count matrix from the raw layer
count_matrix <- adata$raw$X
count_matrix <- t(count_matrix)
count_matrix <- round(count_matrix)

In [5]:
# Define the column data
coldata <- adata$obs[,c("Tumor_Site","Culture_Media", "ZFP_Expression", "Batch", "Patient")]

# Convert column data to factors
coldata$Tumor_Site <- factor(coldata$Tumor_Site)
coldata$Culture_Media <- factor(coldata$Culture_Media)
coldata$ZFP_Expression <- factor(coldata$ZFP_Expression)
coldata$Batch <- factor(coldata$Batch)
coldata$Patient <- factor(coldata$Patient)

In [ ]:
# Design a DESeqDataSet
dds_1 <- DESeqDataSetFromMatrix(
  countData = count_matrix,
  colData = coldata,
  design = ~ Tumor_Site + Culture_Media + Patient + ZFP_Expression + ZFP_Expression:Culture_Media
)

# Add a pseudo-count of 1 to all counts
count_matrix <- counts(dds_1) + 1

# Specify the row names as the gene names
rownames(count_matrix) <- adata$raw$var_names


In [ ]:
# Set the reference levels
dds_1$Tumor_Site<-relevel(dds_1$Tumor_Site,ref="Primary")
dds_1$Culture_Media<-relevel(dds_1$Culture_Media,ref="BASE")
dds_1$ZFP_Expression<-relevel(dds_1$ZFP_Expression,ref="CTRL")
dds_1$Patient<-relevel(dds_1$Patient,ref="125")


In [ ]:
# Design a DESeqDataSet with modified count matrix
dds_1 <- DESeqDataSetFromMatrix(
  countData = count_matrix,
  colData = coldata,
  design = ~ Tumor_Site + Culture_Media + Patient + ZFP_Expression + ZFP_Expression:Culture_Media
)

In [ ]:
# Perform DESeq analysis
dds_1 <- DESeq(dds_1)

In [ ]:
# Extract results
result_names_1 <- resultsNames(dds_1)
result_names_1

# Specify the contrast and extract results
res_1 <- results(dds_1)
res_1

In [ ]:
# Specify the file path where you want to save the results
result_file <- "/data/chanjlab/CRC_ZFP36L2.092023/Organoid/output/deseq/zfpexp_results.csv"

# Write the results table to a CSV file
write.csv(as.data.frame(res_1), file = result_file)

# Save the DESeqDataSet object
saveRDS(dds_1, file = "/data/chanjlab/CRC_ZFP36L2.092023/Organoid/output/deseq/zfpexp_dds.rds")

# Save the DESeq results
saveRDS(results(dds_1), file = "/data/chanjlab/CRC_ZFP36L2.092023/Organoid/output/deseq/zfpexp_results.rds")